In [1]:
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np

In [2]:
df = pd.read_csv("healthcare_dataset_with_los.csv")

In [3]:
import statsmodels.formula.api as smf

# 1. Define the formula (Y ~ X)
# We are predicting 'Billing Amount' based on 'Length of Stay'
formula = 'Q("Billing Amount") ~ Q("Length of Stay")'

# 2. Build and fit the model
model = smf.ols(formula=formula, data=df)
results = model.fit()

# 3. Print the statistical summary
print(results.summary())

                             OLS Regression Results                            
Dep. Variable:     Q("Billing Amount")   R-squared:                       0.000
Model:                             OLS   Adj. R-squared:                  0.000
Method:                  Least Squares   F-statistic:                     1.742
Date:                 Sat, 18 Apr 2026   Prob (F-statistic):              0.187
Time:                         19:14:30   Log-Likelihood:            -6.0943e+05
No. Observations:                55500   AIC:                         1.219e+06
Df Residuals:                    55498   BIC:                         1.219e+06
Df Model:                            1                                         
Covariance Type:             nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept            2.5

In [ ]:
# Absolutely no correlation between length of stay and billing amount. The p-value is very high, and the R-squared value is very low, indicating that the model does not explain any of the variability in billing amount based on length of stay. Let's try Multiple Linear Regression by including more variables.

In [4]:
import statsmodels.formula.api as smf

# 1. Define the Multiple Regression formula
# Y ~ X1 + X2 + X3...
# Note: Use Q("Column Name") if the name has spaces!
formula = 'Q("Billing Amount") ~ Q("Length of Stay") + Age + Q("Room Number")'

# 2. Build and fit the model
model = smf.ols(formula=formula, data=df)
results = model.fit()

# 3. Print the massive statistical summary
print(results.summary())

                             OLS Regression Results                            
Dep. Variable:     Q("Billing Amount")   R-squared:                       0.000
Model:                             OLS   Adj. R-squared:                  0.000
Method:                  Least Squares   F-statistic:                     1.010
Date:                 Sat, 18 Apr 2026   Prob (F-statistic):              0.387
Time:                         19:20:18   Log-Likelihood:            -6.0943e+05
No. Observations:                55500   AIC:                         1.219e+06
Df Residuals:                    55496   BIC:                         1.219e+06
Df Model:                            3                                         
Covariance Type:             nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept            2.5

In [ ]:
# There is no significant relationship between length of stay, age, room number and billing amount. The p-values for all the predictors are very high, and the R-squared value is still very low, indicating that the model does not explain any of the variability in billing amount based on these predictors. We may need to consider other variables or a different modeling approach to better understand the factors influencing billing amount. Lets try Logistic Regression by converting billing amount into a binary variable (e.g., high vs low billing amount) and see if we can predict that based on the other variables.

In [ ]:
import statsmodels.formula.api as smf
import numpy as np

# 1. Create our binary target variable (1 if Emergency, 0 if anything else)
df["Is_Emergency"] = np.where(df["Admission Type"] == "Emergency", 1, 0)

# 2. Define the formula (Y ~ X1 + X2)
formula = 'Is_Emergency ~ Q("Billing Amount") + Q("Length of Stay")'

# 3. Build and fit the LOGISTIC model
# Notice we use smf.logit() instead of smf.ols()
model = smf.logit(formula=formula, data=df)
results = model.fit()

# 4. Print the summary
print(results.summary())

Optimization terminated successfully.
         Current function value: 0.633564
         Iterations 4
                           Logit Regression Results                           
Dep. Variable:           Is_Emergency   No. Observations:                55500
Model:                          Logit   Df Residuals:                    55497
Method:                           MLE   Df Model:                            2
Date:                Sat, 18 Apr 2026   Pseudo R-squ.:               4.147e-05
Time:                        20:07:52   Log-Likelihood:                -35163.
converged:                       True   LL-Null:                       -35164.
Covariance Type:            nonrobust   LLR p-value:                    0.2326
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept              -0.7307      0.025    -29.564      0.000      -0.779      -0.682
Q(

In [ ]:
import pandas as pd
import numpy as np

# 1. Create a "New Patient" to test the model
# Let's say they stayed 12 days and have a $25,000 bill
new_patients = pd.DataFrame(
    {"Length of Stay": [12, 2], "Billing Amount": [25000, 1500]}
)

# 2. Ask the model to predict the PROBABILITY of an Emergency (Outputs 0 to 1)
probabilities = results.predict(new_patients)

# 3. Convert probabilities into hard classifications (1 or 0)
# We use 0.5 (50%) as our cutoff threshold
predictions = np.where(probabilities >= 0.5, 1, 0)

print("Probabilities:")
print(probabilities)
print("\nFinal Classifications (1=Emergency, 0=Other):")
print(predictions)

Probabilities:
0    0.327876
1    0.325689
dtype: float64

Final Classifications (1=Emergency, 0=Other):
[0 0]


In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd
import numpy as np

# 1. Ask the model to predict probabilities for the entire dataset
all_probabilities = results.predict(df)

# 2. Convert those probabilities to 1s and 0s (using our 50% threshold)
all_predictions = np.where(all_probabilities >= 0.5, 1, 0)

# 3. Create the Confusion Matrix
# We compare the actual 'Is_Emergency' column to our 'all_predictions'
cm = confusion_matrix(df["Is_Emergency"], all_predictions)

# 4. Print it out nicely using pandas
cm_df = pd.DataFrame(
    cm,
    index=["Actual Not Emergency (0)", "Actual Emergency (1)"],
    columns=["Predicted Not Emergency (0)", "Predicted Emergency (1)"],
)

print("CONFUSION MATRIX:")
print("-" * 50)
print(cm_df)

CONFUSION MATRIX:
--------------------------------------------------
                          Predicted Not Emergency (0)  Predicted Emergency (1)
Actual Not Emergency (0)                        37231                        0
Actual Emergency (1)                            18269                        0


In [ ]:
# Logistic Regression is not performing well in predicting Emergency admissions based on Billing Amount and Length of Stay. The confusion matrix shows that the model is mostly predicting "Not Emergency" (0) and very few "Emergency" (1), which indicates that it may not be capturing the underlying patterns in the data effectively. We may need to consider additional features, feature engineering, or a different modeling approach to improve the predictive performance.